# FRA Already-On-BFP DC Scope

Converts the FRA **Already_On_BFP** scope CSV into the Data Correctness (DC) run format.
These are APTs that already sit on the correct building footprint, so the DC run only
marks them `ALREADY_ON_CORRECT_BFP`. It sends no target value or address component.

Reference: [SEACO-6799](https://tomtom.atlassian.net/browse/SEACO-6799)

## What this notebook does

1. **Read the input CSV** with pandas. `Product_orbis_id` is read as text so the long composite id survives
2. **Map columns** to the DC run format (see the table below)
3. **Write the output CSV** `SEACO-6799-FRA-Already-On-BFP.csv`, next to the input file

## Input columns

`cntry_code, Product_orbis_id, address_point_type, Orbis_X_OLD, Orbis_Y_OLD, old_location_xy,
Orbis_X_New, Orbis_Y_New, new_location_xy, new_address_point_type, Product_Address_component, Relocation_Type`

## Column mapping

| Output column | Source |
|---|---|
| `Product_orbis_id` | `Product_orbis_id` |
| `Product_Address_component` | empty string |
| `target_value` | empty string |
| `Orbis_X` | `Orbis_Y_New`, at 7 decimals |
| `Orbis_Y` | `Orbis_X_New`, at 7 decimals |
| `Relocation_Type` | constant `ALREADY_ON_CORRECT_BFP` |

`Orbis_X` takes `Orbis_Y_New` and `Orbis_Y` takes `Orbis_X_New`. The lat/lng order is swapped
on purpose, the same as in the `Need_to_relocate` scope split (`RoofTop-FRA-Scoping.ipynb`).
Orbis stores coordinates as integers at 1e-7 degree precision, so each value is snapped to that
grid and written with exactly 7 decimals: no truncation and no scientific notation.

In [1]:
import pandas as pd

INPUT_CSV = "/Users/sawan.darekar/Desktop/workspace_data/SEACO-6799-FRA-Already-On-BFP/part-00000-tid-4396075724317608784-63acf2bb-1f15-4028-bf12-41bdcf5c6ae2-1787-1-c000.csv"

# Product_orbis_id is a long composite id -> keep it as text
input_df = pd.read_csv(INPUT_CSV, dtype={"Product_orbis_id": str})

input_df.head()

,cntry_code,Product_orbis_id,address_point_type,Orbis_X_OLD,Orbis_Y_OLD,old_location_xy,Orbis_X_New,Orbis_Y_New,new_location_xy,new_address_point_type,Product_Address_component,Relocation_Type
0,FRA,14533_11526378061407780864_17366895294335249826,land_parcel,-0.554345,42.871758,POINT (-0.5543449 42.871758),-0.554324,42.871762,POINT (-0.5543237025023027 42.87176151960548),building,metadata:apa:previous_location,BUILDING_FOOTPRINT
1,FRA,14533_11526388734664704000_12833607080635312519,land_parcel,-0.009420,42.731772,POINT (-0.00942 42.731772),-0.009420,42.731772,POINT (-0.00942 42.731772082942065),building,metadata:apa:previous_location,BUILDING_FOOTPRINT
2,FRA,14533_11526399118247526400_982833700341554188,building,-0.113181,42.887443,POINT (-0.113181 42.887443),-0.113197,42.887440,POINT (-0.1131972105175773 42.88744046157208),building,metadata:apa:previous_location,BUILDING_FOOTPRINT
3,FRA,14533_11526529367948984320_10481048846173583703,building,-1.439404,43.081656,POINT (-1.439404 43.081656),-1.439404,43.081656,POINT (-1.439404 43.08165608316952),building,metadata:apa:previous_location,BUILDING_FOOTPRINT
4,FRA,14533_11526626233392889856_14904865842700941475,building,-1.761102,43.346560,POINT (-1.761102 43.34656),-1.761097,43.346554,POINT (-1.7610972709086765 43.34655352123956),building,metadata:apa:previous_location,BUILDING_FOOTPRINT


In [2]:
ALREADY_ON_CORRECT_BFP = "ALREADY_ON_CORRECT_BFP"

# Orbis stores coordinates as integers at 1e-7 degree precision
DEFAULT_SCALE = 1.0e7

OUTPUT_COLUMNS = [
    "Product_orbis_id",
    "Product_Address_component",
    "target_value",
    "Orbis_X",
    "Orbis_Y",
    "Relocation_Type",
]


def to_orbis_coordinate(series):
    """Snap to the Orbis 1e-7 grid and render as a fixed 7-decimal string."""
    snapped = (series.astype(float) * DEFAULT_SCALE).round() / DEFAULT_SCALE
    return snapped.map(lambda value: f"{value:.7f}")


# Note: Orbis_X takes Orbis_Y_New and Orbis_Y takes Orbis_X_New (lat/lng order is swapped on purpose).
scope_df = pd.DataFrame({
    "Product_orbis_id": input_df["Product_orbis_id"],
    "Product_Address_component": "",
    "target_value": "",
    "Orbis_X": to_orbis_coordinate(input_df["Orbis_Y_New"]),
    "Orbis_Y": to_orbis_coordinate(input_df["Orbis_X_New"]),
    "Relocation_Type": ALREADY_ON_CORRECT_BFP,
})[OUTPUT_COLUMNS]

scope_df.head()

,Product_orbis_id,Product_Address_component,target_value,Orbis_X,Orbis_Y,Relocation_Type
0,14533_11526378061407780864_17366895294335249826,,,42.8717615,-0.5543237,ALREADY_ON_CORRECT_BFP
1,14533_11526388734664704000_12833607080635312519,,,42.7317721,-0.0094200,ALREADY_ON_CORRECT_BFP
2,14533_11526399118247526400_982833700341554188,,,42.8874405,-0.1131972,ALREADY_ON_CORRECT_BFP
3,14533_11526529367948984320_10481048846173583703,,,43.0816561,-1.4394040,ALREADY_ON_CORRECT_BFP
4,14533_11526626233392889856_14904865842700941475,,,43.3465535,-1.7610973,ALREADY_ON_CORRECT_BFP


In [3]:
OUTPUT_DIR = "/Users/sawan.darekar/Desktop/workspace_data/SEACO-6799-FRA-Already-On-BFP/"
OUTPUT_CSV = OUTPUT_DIR + "SEACO-6799-FRA-Already-On-BFP.csv"

scope_df.to_csv(OUTPUT_CSV, index=False)

In [4]:
print(f"Input rows   : {len(input_df)}")
print(f"Output rows  : {len(scope_df)}")
print(f"Output file  : {OUTPUT_CSV}")

Input rows   : 2828067
Output rows  : 2828067
Output file  : /Users/sawan.darekar/Desktop/workspace_data/SEACO-6799-FRA-Already-On-BFP/SEACO-6799-FRA-Already-On-BFP.csv
